In [33]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [34]:
movies_df = pd.read_csv("./ml-32m/movies.csv")
ratings_df=pd.read_csv("./ml-32m/ratings.csv")
links_df=pd.read_csv("./ml-32m/links.csv")

In [35]:
movies_df.head(5)

,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy


In [36]:
links_df.head(5)

,movieId,imdbId,tmdbId
0,1,114709,862.0
1,2,113497,8844.0
2,3,113228,15602.0
3,4,114885,31357.0
4,5,113041,11862.0


In [37]:
ratings_df.head(5)

,userId,movieId,rating,timestamp
0,1,17,4.0,944249077
1,1,25,1.0,944250228
2,1,29,2.0,943230976
3,1,30,5.0,944249077
4,1,32,5.0,943228858


In [38]:
print(movies_df.shape)
print(ratings_df.shape)
print(links_df.shape)

(87585, 3)
(32000204, 4)
(87585, 3)


In [39]:
ratings_df["userId"][len(ratings_df)-1].item()

200948

In [40]:
def movieRating(movie):
    mask=movies_df["title"]==movie
    movie=movies_df[mask]["movieId"].item()
    mask=ratings_df["movieId"]==movie
    return np.array(ratings_df[mask]["rating"]).mean().item()

movieRating("Toy Story (1995)")



3.8974375697494095

In [41]:
small_movies_df=movies_df.head(100)
small_movies_df["genres"]=small_movies_df["genres"].str.split("|")
small_movies_df["averageRating"]=small_movies_df["title"].apply(movieRating)
small_movies_df

,movieId,title,genres,averageRating
0,1,Toy Story (1995),"[Adventure, Animation, Children, Comedy, Fantasy]",3.897438
1,2,Jumanji (1995),"[Adventure, Children, Fantasy]",3.275758
2,3,Grumpier Old Men (1995),"[Comedy, Romance]",3.139447
3,4,Waiting to Exhale (1995),"[Comedy, Drama, Romance]",2.845331
4,5,Father of the Bride Part II (1995),[Comedy],3.059602
...,...,...,...,...
95,97,"Hate (Haine, La) (1995)","[Crime, Drama]",4.004987
96,98,Shopping (1994),"[Action, Thriller]",2.614130
97,99,Heidi Fleiss: Hollywood Madam (1995),[Documentary],3.089481
98,100,City Hall (1996),"[Drama, Thriller]",3.218223


In [42]:
small_movies_df["genres"]
list_of_genres=small_movies_df["genres"].explode().unique()
list(list_of_genres)


['Adventure',
 'Animation',
 'Children',
 'Comedy',
 'Fantasy',
 'Romance',
 'Drama',
 'Action',
 'Crime',
 'Thriller',
 'Horror',
 'Mystery',
 'Sci-Fi',
 'IMAX',
 'Documentary',
 'War',
 'Musical']

In [43]:
def createIDF():
    N = len(small_movies_df)

    df_count = np.zeros(len(list_of_genres))
    for i, genre in enumerate(list_of_genres):

        # Same as below
        sum = 0
        for genres in small_movies_df["genres"]:
            sum += genre in genres
        df_count[i] = sum

        # Same as above
        df_count[i] = small_movies_df["genres"].apply(lambda gs: genre in gs).sum()

    # Inverse of Data Frequency
        idf = np.log(N / (df_count + 1))
        return idf

idf_vectors=(createIDF())

def createVector(genres):
    vector=np.zeros(len(list_of_genres))
    for i in range(0,len(list_of_genres)):
        vector[i]=int(list_of_genres[i] in genres)
    return np.array(vector)

def create_tfidf_vector(genres):
    onehot = createVector(genres)
    tf = onehot / len(genres)
    return tf * idf_vectors

create_tfidf_vector(["Adventure","Musical","Crime"])

array([0.61086049, 0.        , 0.        , 0.        , 0.        ,
       0.        , 0.        , 0.        , 1.53505673, 0.        ,
       0.        , 0.        , 0.        , 0.        , 0.        ,
       0.        , 1.53505673])

# hello 
*text* 

In [48]:





        



small_movies_df["onehot"]=(small_movies_df["genres"]).apply(createVector)
small_movies_df["tf_idf"]=(small_movies_df["genres"]).apply(create_tfidf_vector)
small_movies_df

,movieId,title,genres,averageRating,onehot,tf_idf
0,1,Toy Story (1995),"[Adventure, Animation, Children, Comedy, Fantasy]",3.897438,"[1.0, 1.0, 1.0, 1.0, 1.0, 0.0, 0.0, 0.0, 0.0, ...","[0.3665162927496621, 0.9210340371976184, 0.921..."
1,2,Jumanji (1995),"[Adventure, Children, Fantasy]",3.275758,"[1.0, 0.0, 1.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, ...","[0.6108604879161034, 0.0, 1.5350567286626973, ..."
2,3,Grumpier Old Men (1995),"[Comedy, Romance]",3.139447,"[0.0, 0.0, 0.0, 1.0, 0.0, 1.0, 0.0, 0.0, 0.0, ...","[0.0, 0.0, 0.0, 2.302585092994046, 0.0, 2.3025..."
3,4,Waiting to Exhale (1995),"[Comedy, Drama, Romance]",2.845331,"[0.0, 0.0, 0.0, 1.0, 0.0, 1.0, 1.0, 0.0, 0.0, ...","[0.0, 0.0, 0.0, 1.5350567286626973, 0.0, 1.535..."
4,5,Father of the Bride Part II (1995),[Comedy],3.059602,"[0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.0, 0.0, 0.0, 4.605170185988092, 0.0, 0.0, 0..."
...,...,...,...,...,...,...
95,97,"Hate (Haine, La) (1995)","[Crime, Drama]",4.004987,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 1.0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 2.3025850929940..."
96,98,Shopping (1994),"[Action, Thriller]",2.614130,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 2.30258509..."
97,99,Heidi Fleiss: Hollywood Madam (1995),[Documentary],3.089481,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."
98,100,City Hall (1996),"[Drama, Thriller]",3.218223,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 2.3025850929940..."


In [45]:
def angleBetweenVectors(a,b):
    dotproduct=np.dot(a,b)
    
    a_length=np.linalg.norm(a)
    b_length=np.linalg.norm(b)
    
    return np.arccos(dotproduct/(a_length*b_length))
angleBetweenVectors(np.array([2,2]),np.array([2,0]))

np.float64(0.7853981633974484)

In [50]:
def topKMovies(movie,top_k=5):
    mask=small_movies_df["title"]==movie
    vector=small_movies_df[mask]["tf_idf"].item()
    angle_list=[]
    for i,row in small_movies_df.iterrows():
        
        angle_list.append([angleBetweenVectors(vector,row["tf_idf"]),row["title"],row["genres"]])
    angle_list.sort()
    return angle_list[0:top_k]


topKMovies("Toy Story (1995)",top_k=5)

[[np.float64(1.4901161193847656e-08),
  'Toy Story (1995)',
  ['Adventure', 'Animation', 'Children', 'Comedy', 'Fantasy']],
 [np.float64(0.5125340223930298),
  'Kids of the Round Table (1995)',
  ['Adventure', 'Children', 'Comedy', 'Fantasy']],
 [np.float64(0.7663528314695585),
  'Balto (1995)',
  ['Adventure', 'Animation', 'Children']],
 [np.float64(0.7663528314695585),
  'Indian in the Cupboard, The (1995)',
  ['Adventure', 'Children', 'Fantasy']],
 [np.float64(0.7663528314695585),
  'Jumanji (1995)',
  ['Adventure', 'Children', 'Fantasy']]]